### 循环神经网络RNN（Recurrent Neural Network）
RNN与CNN最大的不同在于CNN只关注单独一个输入，每次输入之间没有关系，而RNN的输入往往是一个序列，当前时刻的状态，不仅取决于当前输入，还取决于之前的信息，即
$$h_t = f_W(h_{t-1},x_t)$$
其中$h_t$是新的状态（state），$h_{t-1}$是过去的状态，$x_t$是$t$时刻新的输入，$f_W$是以$W$为参数的某个函数（例如全连接神经网络中对输入乘以矩阵$W$并用激活函数作用）。以上称为RNN隐藏状态的更新，$f_w$称为一个循环函数（Recurrence Fomula），它在每次循环时是**相同**的（即函数、参数、激活函数相同）。对于每个状态$h_t$，我们又通过另外一个函数（参数不同，为$W_{hy}$，可能没有激活函数）计算输出$y_t$，即
$$y_t = f_{W_{hy}}(h_t)$$
<div align="center">
  <img src="class_images/unrolled_RNN.jpg" width="800">
</div>

In [ ]:
"""
这是一个展示RNN原理的示例，其作用是对于输入序列x_seq，
若当前输入和上一个输入均为1，则输出为1，否则输出为0
"""
# 这个算法中，中间状态h_t是一个三维向量，分别表示当前输入x、上一个状态h_t_prev，以及一个常数项1
import torch

w_xh = torch.tensor([1., 0., 0.])   # 将当前输入x记录在中间状态的第一个位置
w_hh = torch.tensor([[0., 0., 0.],  # 将上一个状态h_t_prev记录在中间状态的第二个位置，并确保第三个位置始终为1
                     [1., 0., 0.],
                     [0., 0., 1.]])
w_hy = torch.tensor([1., 1., -1.])
x_seq = [0, 1, 0, 1, 1, 1, 0, 1, 1]
y_seq = []
h_t_prev = torch.tensor([0., 0., 1.])

for t, x in enumerate(x_seq):
    h_t = torch.relu(w_xh * x + w_hh @ h_t_prev)
    y_t = torch.relu(w_hy @ h_t)
    h_t_prev = h_t
    y_seq.append(y_t.item())

print(y_seq)

[0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0]


#### 梯度的计算
Backpropagation through time：前向穿过整个序列计算损失，反向穿过整个序列计算梯度。这样做的一个问题是可能导致内存不足（如果只有一个输入，假想后面每个隐藏状态的输入是0）
<div align="center">
  <img src="class_images/RNN梯度的计算.jpg" width="500">
</div>

Truncated Backpropagation through time：前向穿过整个序列，但在反向传播时，只穿过整个序列的一个chunk，减少计算量（如果只有一个输出，则只关心RNN的最后一部分序列）

* RNN常用于语言模型，对于字符，通常采用独热编码，但作为输入前有一个嵌入层（Embedding Layer）；实践中，通常人为加入的一个特殊token，用来告诉RNN句子从这里开始生成。因为生成第一个词之前没有真实输入词，所以需要用START作为第一个输入

#### RNN的优势与缺陷
优点：输入文字的长度是任意的；理论上某一步的计算可以使用之前很多步的信息；输入变长并不会导致模型规模变大；每一步的权重是相同的<br>
缺点：循环计算通常缓慢；实践中，由于中间状态的大小是固定的，我们无法将所有之前的信息塞进去，因此实际上并不能使用之前很多步的信息

#### RNN在视觉领域的应用
* 图片注释（Image Captioning）：先由CNN提取出特征向量，然后作为输入的一部分喂给RNN，RNN结合学到的词语预测来为它写注释，即
$$h=tanh(W_{xh}*​x+W_{hh}*​h+W{ih}*​v)$$
* Visual Question Answering

#### RNN的变体
多层RNN、LSTM（Long Short Term Memory）：Gradient Flow（用于解决一般RNN计算梯度时梯度消失等问题，它的想法有点像ResNet）
